In [3]:
import sys, inspect
sys.path.append("..")
from src import dataset
print(inspect.getsource(dataset.gather_samples))
print(inspect.signature(dataset.RiceLeafDataset.__init__))
print(inspect.signature(dataset.get_transforms))
print(inspect.signature(dataset.compute_class_weights))

def gather_samples():
    """Return list of (filepath, label_idx) for all 7 classes."""
    samples = []
    for cls in CLASSES:
        cls_dir = DATA_DIR / cls
        for f in cls_dir.iterdir():
            if f.suffix.lower() in VALID_EXT:
                samples.append((str(f), CLASS_TO_IDX[cls]))
    return samples

(self, samples, split='train')
(split='train')
(train_set)


In [7]:
import sys, time, copy, json, numpy as np, torch, torch.nn as nn, torch.optim as optim
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader
from torchvision import models
sys.path.append("../src")
from dataset import gather_samples, compute_class_weights, RiceLeafDataset, CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FOLDS, SEED, BATCH_SIZE, LR = 5, 42, 32, 1e-4
NUM_EPOCHS, PATIENCE, NUM_WORKERS = 30, 7, 4
OUT = Path("../results/cv"); OUT.mkdir(parents=True, exist_ok=True)

samples = gather_samples()
labels = np.array([s[1] for s in samples])
NUM_CLASSES = len(CLASSES)
print(device, len(samples), NUM_CLASSES, dict(zip(CLASSES, np.bincount(labels))))

cuda 2753 7 {'Healthy': np.int64(603), 'Insect': np.int64(281), 'Leaf Scald': np.int64(421), 'Rice Blast': np.int64(696), 'Rice Leaffolder': np.int64(247), 'Rice Stripes': np.int64(266), 'Rice Tungro': np.int64(239)}


In [8]:
def make_model():
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES)
    return m.to(device)

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, y in loader:
            imgs, y = imgs.to(device), y.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs); loss = criterion(out, y)
            if train: loss.backward(); optimizer.step()
            loss_sum += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == y).sum().item(); total += y.size(0)
    return loss_sum / total, correct / total

def predict(model, loader):
    model.eval(); preds, gts = [], []
    with torch.no_grad():
        for imgs, y in loader:
            preds += model(imgs.to(device)).argmax(1).cpu().tolist(); gts += y.tolist()
    return gts, preds

def run_fold(fold, tr_idx, te_idx):
    y_tr = labels[tr_idx]
    tr_i, va_i = train_test_split(tr_idx, test_size=0.1765, stratify=y_tr, random_state=SEED)
    tr_set = [samples[i] for i in tr_i]
    va_set = [samples[i] for i in va_i]
    te_set = [samples[i] for i in te_idx]

    mk = lambda ds, sp, sh: DataLoader(RiceLeafDataset(ds, split=sp), batch_size=BATCH_SIZE,
                                       shuffle=sh, num_workers=NUM_WORKERS, pin_memory=True)
    tl, vl, el = mk(tr_set, "train", True), mk(va_set, "val", False), mk(te_set, "test", False)

    model = make_model()
    w = torch.tensor(compute_class_weights(tr_set), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=w)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    best_loss, best_wts, bad = float("inf"), copy.deepcopy(model.state_dict()), 0
    for ep in range(NUM_EPOCHS):
        trl, tra = run_epoch(model, tl, criterion, optimizer)
        vl_loss, vl_acc = run_epoch(model, vl, criterion)
        scheduler.step(vl_loss)
        print(f"fold{fold} ep{ep+1} train_acc={tra:.4f} val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}")
        if vl_loss < best_loss:
            best_loss, best_wts, bad = vl_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE: print("early stop"); break

    model.load_state_dict(best_wts)
    gts, preds = predict(model, el)
    del model; torch.cuda.empty_cache()
    return {"fold": fold,
            "macro_f1": f1_score(gts, preds, average="macro"),
            "acc": accuracy_score(gts, preds),
            "per_class_f1": f1_score(gts, preds, average=None, labels=list(range(NUM_CLASSES))).tolist()}

In [6]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
results = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    t0 = time.time()
    r = run_fold(fold, tr_idx, va_idx); results.append(r)
    json.dump(results, open(OUT / "cv_results.json", "w"))
    print(f"== Fold {fold}: F1={r['macro_f1']:.4f} Acc={r['acc']:.4f} ({(time.time()-t0)/60:.1f} min)")

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Error while downloading from https://us.aws.cdn.hf.co/xet-bridge-us/63bf39ae20784381e8ead18e/6e0a341b439af9b46ceddfd874c838fb3af7b0faa10044c438cbfa93bdf23023?X-Xet-Cas-Uid=public&xip=rrTZ4oPeU_k&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1790702103&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjNiZjM5YWUyMDc4NDM4MWU4ZWFkMThlLzZlMGEzNDFiNDM5YWY5YjQ2Y2VkZGZkODc0YzgzOGZiM2FmN2IwZmFhMTAwNDRjNDM4Y2JmYTkzYmRmMjMwMjNcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9cnJUWjRvUGVVX2smdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj1pbmxpbmUlM0IrZmlsZW5hbWUlMkElM0RVVEYtOCUyNyUyN21vZGVsLnNhZmV0ZW5zb3JzJTNCK2ZpbGVuYW1lJTNEJTIybW9kZWwuc2FmZXRlbnNvcnMlMjIlM0IiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc5MDcwMjEwM319fV19&Signature=MEYCIQC8NbYc8A9HgeOJNoyjveVTqkUYpUyjGgGJpMxwLX7s5QIhANoaT8H39UrdXNiquYR4ZaPnoaHt7EkoSQj1rqooCWGW&Key-Pair-Id=01K

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Error while downloading from https://us.aws.cdn.hf.co/xet-bridge-us/63bf39ae20784381e8ead18e/6e0a341b439af9b46ceddfd874c838fb3af7b0faa10044c438cbfa93bdf23023?X-Xet-Cas-Uid=public&xip=rrTZ4oPeU_k&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1790702103&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjNiZjM5YWUyMDc4NDM4MWU4ZWFkMThlLzZlMGEzNDFiNDM5YWY5YjQ2Y2VkZGZkODc0YzgzOGZiM2FmN2IwZmFhMTAwNDRjNDM4Y2JmYTkzYmRmMjMwMjNcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9cnJUWjRvUGVVX2smdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj1pbmxpbmUlM0IrZmlsZW5hbWUlMkElM0RVVEYtOCUyNyUyN21vZGVsLnNhZmV0ZW5zb3JzJTNCK2ZpbGVuYW1lJTNEJTIybW9kZWwuc2FmZXRlbnNvcnMlMjIlM0IiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc5MDcwMjEwM319fV19&Signature=MEYCIQC8NbYc8A9HgeOJNoyjveVTqkUYpUyjGgGJpMxwLX7s5QIhANoaT8H39UrdXNiquYR4ZaPnoaHt7EkoSQj1rqooCWGW&Key-Pair-Id=01K

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Error while downloading from https://us.aws.cdn.hf.co/xet-bridge-us/63bf39ae20784381e8ead18e/6e0a341b439af9b46ceddfd874c838fb3af7b0faa10044c438cbfa93bdf23023?X-Xet-Cas-Uid=public&xip=rrTZ4oPeU_k&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1790702103&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjNiZjM5YWUyMDc4NDM4MWU4ZWFkMThlLzZlMGEzNDFiNDM5YWY5YjQ2Y2VkZGZkODc0YzgzOGZiM2FmN2IwZmFhMTAwNDRjNDM4Y2JmYTkzYmRmMjMwMjNcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9cnJUWjRvUGVVX2smdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj1pbmxpbmUlM0IrZmlsZW5hbWUlMkElM0RVVEYtOCUyNyUyN21vZGVsLnNhZmV0ZW5zb3JzJTNCK2ZpbGVuYW1lJTNEJTIybW9kZWwuc2FmZXRlbnNvcnMlMjIlM0IiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc5MDcwMjEwM319fV19&Signature=MEYCIQC8NbYc8A9HgeOJNoyjveVTqkUYpUyjGgGJpMxwLX7s5QIhANoaT8H39UrdXNiquYR4ZaPnoaHt7EkoSQj1rqooCWGW&Key-Pair-Id=01K

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Error while downloading from https://us.aws.cdn.hf.co/xet-bridge-us/63bf39ae20784381e8ead18e/6e0a341b439af9b46ceddfd874c838fb3af7b0faa10044c438cbfa93bdf23023?X-Xet-Cas-Uid=public&xip=rrTZ4oPeU_k&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1790702103&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjNiZjM5YWUyMDc4NDM4MWU4ZWFkMThlLzZlMGEzNDFiNDM5YWY5YjQ2Y2VkZGZkODc0YzgzOGZiM2FmN2IwZmFhMTAwNDRjNDM4Y2JmYTkzYmRmMjMwMjNcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9cnJUWjRvUGVVX2smdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj1pbmxpbmUlM0IrZmlsZW5hbWUlMkElM0RVVEYtOCUyNyUyN21vZGVsLnNhZmV0ZW5zb3JzJTNCK2ZpbGVuYW1lJTNEJTIybW9kZWwuc2FmZXRlbnNvcnMlMjIlM0IiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc5MDcwMjEwM319fV19&Signature=MEYCIQC8NbYc8A9HgeOJNoyjveVTqkUYpUyjGgGJpMxwLX7s5QIhANoaT8H39UrdXNiquYR4ZaPnoaHt7EkoSQj1rqooCWGW&Key-Pair-Id=01K

ConnectionError: HTTPSConnectionPool(host='us.aws.cdn.hf.co', port=443): Read timed out.